# SPATIAL INTELLIGENCE — PART 3
## Homework 04 — ETM

## 1. Import the needed libraries

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy Version

In [ ]:
print("This notebook requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

## 3. Set your renderer
* Visual Studio Code: `"vscode"`
* Google Colab: `"colab"`
* Browser: `"browser"`

In [ ]:
renderer = "vscode"

## 4. Utility functions

In [ ]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == 'face_id':
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts.get(str(value), None)
            if f:
                f = Topology.SetDictionary(f, d)

## 5. Load the floor plan outline (OBJ)

In [ ]:
FLOOR_PLANS = r"C:\Users\etmaglari\IAAC\etmaglari_gML\Homework04\FloorPlans"
OBJ_PATH = FLOOR_PLANS + r"\GroungFloor.obj"

objects = Topology.ByOBJPath(OBJ_PATH)
if not isinstance(objects, list): objects = [objects]

# Ground floor OBJ contains a single large up-facing polygon — the building footprint
floor_outline = None
best_area = 0
for obj in objects:
    for f in (Topology.Faces(obj) or []):
        n = Face.Normal(f)
        if n[2] > 0.7:
            a = Face.Area(f)
            if a and a > best_area:
                best_area = a
                floor_outline = f

print("Floor outline loaded:", floor_outline is not None)

b_r = Wire.BoundingRectangle(floor_outline)
d_br = Topology.Dictionary(b_r)
xmin   = Dictionary.ValueAtKey(d_br, "xmin")
xmax   = Dictionary.ValueAtKey(d_br, "xmax")
ymin   = Dictionary.ValueAtKey(d_br, "ymin")
ymax   = Dictionary.ValueAtKey(d_br, "ymax")
width  = Dictionary.ValueAtKey(d_br, "width")
length = Dictionary.ValueAtKey(d_br, "length")

print(f"Bounds: x=[{xmin:.2f}, {xmax:.2f}]  y=[{ymin:.2f}, {ymax:.2f}]")
print(f"Size: {width:.2f} x {length:.2f} m")

# All four floors share the same footprint; translate to each floor elevation
FLOOR_LEVELS = [
    ("Ground Floor",  0.000),
    ("Level 1",       2.079),
    ("Level 2",       5.710),
    ("Level 3",       8.483),
]
floor_faces = []
for name, z in FLOOR_LEVELS:
    ff = Topology.Translate(floor_outline, 0, 0, z)
    floor_faces.append((name, z, ff))
    print(f"  {name}: z={z:.3f}")

## 6. Show the floor plans

In [ ]:
for name, z, ff in floor_faces:
    fc = Topology.Centroid(ff)
    cx, cy, cz = Vertex.X(fc), Vertex.Y(fc), Vertex.Z(fc)
    print(f"--- {name} ---")
    Topology.Show(ff,
                  camera=[0, 0, 6],
                  faceColor=[210, 210, 250], faceOpacity=1,
                  edgeColor="white", edgeWidth=3,
                  showVertices=False,
                  backgroundColor="black",
                  width=800, height=600, renderer=renderer)

## 7. Build grid, shell, and analysis graph for each floor

In [ ]:
GRID_STEP = 1
uRange = list(range(0, int(width)  + GRID_STEP, GRID_STEP))
vRange = list(range(0, int(length) + GRID_STEP, GRID_STEP))

shells, faces_list, g_verts_list = [], [], []
nav_graphs, ana_graphs = [], []

for name, z, ff in floor_faces:
    grid  = Grid.EdgesByDistances(ff, clip=True, uRange=uRange, vRange=vRange)
    shell = Topology.Slice(ff, grid)
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        Topology.SetDictionary(f, Dictionary.ByKeyValue("face_id", "face_" + str(i + 1)))
    nav_g = Graph.ByTopology(shell, direct=False, viaSharedTopologies=True)
    ana_g = Graph.ByTopology(shell)
    g_v   = Graph.Vertices(ana_g)
    shells.append(shell)
    faces_list.append(faces)
    g_verts_list.append(g_v)
    nav_graphs.append(nav_g)
    ana_graphs.append(ana_g)
    print(f"{name}: {len(faces)} cells, {len(g_v)} graph vertices")

## 8. Spatial Intelligence — Isovist / Visibility Analysis
For each floor, isovist polygons are computed from a sparse 2 m viewpoint grid. The number of dense graph vertices visible from each viewpoint is counted, then interpolated onto all shell faces using `Vertex.InterpolateValue`.

In [ ]:
COARSE_STEP = 2   # metres between isovist viewpoints

for i, (name, z, ff) in enumerate(floor_faces):
    fc = Topology.Centroid(ff)
    cx, cy, cz = Vertex.X(fc), Vertex.Y(fc), Vertex.Z(fc)
    print(f"--- {name} — Isovist / Visibility ---")

    iso_face = ff  # whole floor face is the isovist domain

    # Sparse viewpoint grid
    cu = list(range(0, int(width)  + COARSE_STEP, COARSE_STEP))
    cv = list(range(0, int(length) + COARSE_STEP, COARSE_STEP))
    coarse_pts = Grid.VerticesByDistances(ff, uRange=cu, vRange=cv, clip=True)
    iso_verts = [v for v in coarse_pts if Vertex.IsInternal2D(v, iso_face)]
    print(f"  Viewpoints: {len(iso_verts)}")

    # Count visible dense vertices from each viewpoint
    g_v = g_verts_list[i]
    counts = []
    for v in iso_verts:
        iso = Face.Isovist(iso_face, v)
        if iso is None:
            counts.append(0)
            Topology.SetDictionary(v, Dictionary.ByKeysValues(["visibility"], [0]))
            continue
        visible = sum(1 for gv in g_v if Vertex.IsInternal2D(gv, iso))
        counts.append(visible)
        Topology.SetDictionary(v, Dictionary.ByKeysValues(["visibility"], [visible]))

    # Interpolate visibility from sparse to dense vertices
    for gv in g_v:
        Vertex.InterpolateValue(gv, vertices=iso_verts, n=4, key="visibility")

    min_v = min(counts) if counts else 0
    max_v = max(counts) if counts else 1
    for gv in g_v:
        d = Topology.Dictionary(gv)
        vis = Dictionary.ValueAtKey(d, "visibility")
        if vis is not None:
            color = Color.AnyToHex(
                Color.ByValueInRange(vis, minValue=min_v, maxValue=max_v, colorScale="thermal")
            )
            d = Dictionary.SetValueAtKey(d, "vis_color", color)
            Topology.SetDictionary(gv, d)

    reset_dictionaries(shells[i])
    faces_i = Topology.Faces(shells[i])
    transfer_dicts_by_key(faces_i, g_v, "face_id")
    Topology.Show(faces_i,
                  faceColorKey="vis_color", faceOpacity=1,
                  showEdges=False, showVertices=False,
                  camera=[0, 0, 6],
                  backgroundColor="black",
                  width=800, height=600, renderer=renderer)